In [50]:
#!/usr/bin/env python
# coding: utf-8

from pathlib import Path
import os
import re
import time
from datetime import datetime
import json
import hashlib
import random
import shutil
import warnings
from collections import defaultdict
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from llm_sc_curator import LLMscCurator
from llm_sc_curator.backends import GeminiBackend
from llm_sc_curator.masking import FeatureDistiller
from llm_sc_curator.noise_lists import NOISE_PATTERNS, NOISE_LISTS, CELL_CYCLE_GENES

from benchmarks.cd8_config import CD8_HIER_CFG
from benchmarks.hierarchical_scoring import score_hierarchical
from benchmarks.gt_mappings import get_cd8_ground_truth

warnings.filterwarnings("ignore")

In [51]:
# =============================================================================
# 0. Reproducibility / paths / master switches
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

BASE = Path("/work")
INPUT_DIR = BASE / "paper" / "gb_resubmission" / "input"
OUTPUT_DIR = BASE / "paper" / "gb_resubmission" / "output" / "Fig4"
CACHE_DIR = OUTPUT_DIR / "llm_cache
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = INPUT_DIR / "cd8_benchmark_data.h5ad"

# -------------------------------
# Safe-by-default execution plan
# -------------------------------
# Set these to True only when you intentionally want fresh API calls.
RUN_LLM = True
RUN_PANEL_A_LLM = RUN_LLM
RUN_PANEL_C_LLM = RUN_LLM

# Reevaluation / summaries / plots are cheap and deterministic.
RUN_REEVALUATION = True
RUN_PLOTTING = True

# If True, ignore existing CSVs and force fresh generation.
FORCE_RERUN_PANEL_A = False
FORCE_RERUN_PANEL_C = False



In [52]:
# =============================================================================
# 1. .env loader
# =============================================================================
def parse_env_line(line: str):
    s = line.strip()
    if not s or s.startswith("#"):
        return None

    if s.lower().startswith("export "):
        s = s[7:].lstrip()

    if "=" not in s:
        return None

    key, value = s.split("=", 1)
    key = key.strip()

    def strip_inline_comment(val: str) -> str:
        in_single = False
        in_double = False
        for i, ch in enumerate(val):
            if ch == "'" and not in_double:
                in_single = not in_single
            elif ch == '"' and not in_single:
                in_double = not in_double
            elif ch == "#" and not in_single and not in_double:
                return val[:i].rstrip()
        return val

    value = strip_inline_comment(value.strip())

    if (value.startswith('"') and value.endswith('"')) or (value.startswith("'") and value.endswith("'")):
        quote = value[0]
        value = value[1:-1]
        if quote == '"':
            value = (
                value.replace(r"\n", "\n")
                     .replace(r"\r", "\r")
                     .replace(r"\t", "\t")
                     .replace(r"\\", "\\")
                     .replace(r"\"", "\"")
            )
    return key, value


def load_env_file_strict(path: str, override: bool = False):
    if not os.path.exists(path):
        raise FileNotFoundError(f".env not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            parsed = parse_env_line(raw)
            if not parsed:
                continue
            key, value = parsed
            if override:
                os.environ[key] = value
            else:
                os.environ.setdefault(key, value)


load_env_file_strict("/work/.env", override=False)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY and (RUN_PANEL_A_LLM or RUN_PANEL_C_LLM):
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY (input hidden): ")
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]


In [53]:
# =============================================================================
# 2. Backend / curator config
# =============================================================================
ACCURACY_MODE = True
MODEL_NAME = "models/gemini-2.5-pro" if ACCURACY_MODE else "models/gemini-2.0-flash"

# Fig. 4 is a default-workflow necessity figure.
# Standard = raw DEGs baseline (no auto-context)
# LLM-scCurator = default workflow (with default auto-context)
STANDARD_USE_AUTO_CONTEXT_FOR_FIG4 = False
CURATOR_USE_AUTO_CONTEXT_FOR_FIG4 = True

# Conservative scoring rule for reviewer-facing reproducibility:
# use cell_type if available; otherwise fall back to reasoning.
SCORING_TEXT_MODE = "cell_type_primary"  # options: cell_type_primary, cell_type_plus_reasoning

API_SLEEP_SEC = 3.0
SLEEP_STD = API_SLEEP_SEC
SLEEP_CUR = API_SLEEP_SEC
SLEEP_BETWEEN_REPEATS = 3.0


def make_curator(api_key: str, model_name: str):
    try:
        return LLMscCurator(api_key=api_key, model_name=model_name)
    except TypeError:
        backend = GeminiBackend(
            api_key=api_key,
            model_name=model_name,
            temperature=0.0,
        )
        return LLMscCurator(backend=backend)


curator = None
if RUN_PANEL_A_LLM or RUN_PANEL_C_LLM:
    if GEMINI_API_KEY is None or str(GEMINI_API_KEY).strip() == "":
        raise EnvironmentError("GEMINI_API_KEY is not set.")
    curator = make_curator(GEMINI_API_KEY, MODEL_NAME)
    print(f"Using model: {MODEL_NAME}")
    print(f"Standard auto-context = {STANDARD_USE_AUTO_CONTEXT_FOR_FIG4}")
    print(f"Curator auto-context  = {CURATOR_USE_AUTO_CONTEXT_FOR_FIG4}")
    print(f"Scoring text mode     = {SCORING_TEXT_MODE}")
    print(f"curator class         = {curator.__class__.__name__}")
    print(f"curator class         = {curator.__class__.__name__}")


Using model: models/gemini-2.5-pro
Standard auto-context = False
Curator auto-context  = True
Scoring text mode     = cell_type_primary
curator class         = LLMscCurator
curator class         = LLMscCurator


In [54]:
# =============================================================================
# 3. Fig. 4 configuration
# =============================================================================
EXCLUDE_GT_LABELS = {"CD8_Other", "Other", "Unknown"}
N_GENES_LIST = [10, 20, 50, 100, 200]
FAILURE_THRESHOLD = 0.5

# Panel A/B: all eligible CD8 clusters by default
PANEL_A_FOCUS_GT_LABELS = None
PANEL_A_MAX_CLUSTERS_PER_GT = None

# Panel C: prospectively defined multi-cluster stress-test subset
PANEL_C_FOCUS_GT_LABELS = [
    "CD8_Naive",
    "CD8_EffectorMemory",
    "CD8_Exhausted",
    "CD8_Effector",
    "CD8_MAIT",
    "CD8_ISG",
]
PANEL_C_MAX_CLUSTERS_PER_GT = 2
NOISE_RATIOS = (0.0, 0.2, 0.4, 0.6, 0.8)
N_MARKERS_NOISE = 50
N_REPEATS_NOISE = 5

# Panel D: prospectively defined ambiguity-prone states
PANEL_D_HARD_GT_LABELS = [
    "CD8_EffectorMemory",
    "CD8_Effector",
    "CD8_Exhausted",
    "CD8_ISG",
]
PANEL_D_N_GENES = 50

GT_LABEL_DISPLAY = {
    "CD8_Naive": "Naive",
    "CD8_EffectorMemory": "EffMem",
    "CD8_Effector": "Effector",
    "CD8_Exhausted": "Exhausted",
    "CD8_MAIT": "MAIT",
    "CD8_ISG": "ISG",
}

PANEL_A_RAW_CSV = OUTPUT_DIR / "Fig4a_b.csv"
PANEL_C_RAW_CSV = OUTPUT_DIR / "Fig4c.csv"

PANEL_A_REEVAL_CSV = OUTPUT_DIR / "Fig4a_b_REEVALUATED.csv"
PANEL_C_REEVAL_CSV = OUTPUT_DIR / "Fig4c_REEVALUATED.csv"

CONFIG_JSON = OUTPUT_DIR / "Figure4_config.json"

print("DATA_PATH          :", DATA_PATH)
print("OUTPUT_DIR         :", OUTPUT_DIR)
print("PANEL_A_RAW_CSV    :", PANEL_A_RAW_CSV)
print("PANEL_C_RAW_CSV    :", PANEL_C_RAW_CSV)
print("PANEL_A_REEVAL_CSV :", PANEL_A_REEVAL_CSV)
print("PANEL_C_REEVAL_CSV :", PANEL_C_REEVAL_CSV)



DATA_PATH          : /work/paper/gb_resubmission/input/cd8_benchmark_data.h5ad
OUTPUT_DIR         : /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel
PANEL_A_RAW_CSV    : /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelA_input_size_robustness_raw_outputs.csv
PANEL_C_RAW_CSV    : /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelC_multicluster_noise_simulation_raw_outputs.csv
PANEL_A_REEVAL_CSV : /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelA_input_size_robustness_REEVALUATED.csv
PANEL_C_REEVAL_CSV : /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelC_multicluster_noise_simulation_REEVALUATED.csv


In [55]:
config_to_save = {
    "RANDOM_SEED": RANDOM_SEED,
    "MODEL_NAME": MODEL_NAME,
    "STANDARD_USE_AUTO_CONTEXT_FOR_FIG4": STANDARD_USE_AUTO_CONTEXT_FOR_FIG4,
    "CURATOR_USE_AUTO_CONTEXT_FOR_FIG4": CURATOR_USE_AUTO_CONTEXT_FOR_FIG4,
    "SCORING_TEXT_MODE": SCORING_TEXT_MODE,
    "N_GENES_LIST": N_GENES_LIST,
    "FAILURE_THRESHOLD": FAILURE_THRESHOLD,
    "PANEL_A_FOCUS_GT_LABELS": PANEL_A_FOCUS_GT_LABELS,
    "PANEL_A_MAX_CLUSTERS_PER_GT": PANEL_A_MAX_CLUSTERS_PER_GT,
    "PANEL_C_FOCUS_GT_LABELS": PANEL_C_FOCUS_GT_LABELS,
    "PANEL_C_MAX_CLUSTERS_PER_GT": PANEL_C_MAX_CLUSTERS_PER_GT,
    "NOISE_RATIOS": list(NOISE_RATIOS),
    "N_MARKERS_NOISE": N_MARKERS_NOISE,
    "N_REPEATS_NOISE": N_REPEATS_NOISE,
    "PANEL_D_HARD_GT_LABELS": PANEL_D_HARD_GT_LABELS,
    "PANEL_D_N_GENES": PANEL_D_N_GENES,
}
with open(CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(config_to_save, f, ensure_ascii=False, indent=2)

In [56]:
# =============================================================================
# 4. Data loading (only when needed for generation)
# =============================================================================
adata = None

def ensure_hvgs_for_cd8(adata):
    if "highly_variable" in adata.var.columns:
        hv = adata.var["highly_variable"]
        if hv.notna().any():
            print("[OK] highly_variable already present.")
            return

    print("[INFO] highly_variable not found. Computing HVGs once upfront...")

    hvg_kwargs = dict(
        n_top_genes=2000,
        subset=False,
    )

    if "counts" in adata.layers:
        hvg_kwargs["layer"] = "counts"
        hvg_kwargs["flavor"] = "seurat_v3"
    else:
        hvg_kwargs["flavor"] = "seurat"

    if "Cancer_Type" in adata.obs.columns:
        hvg_kwargs["batch_key"] = "Cancer_Type"

    sc.pp.highly_variable_genes(adata, **hvg_kwargs)
    n_hvg = int(np.nansum(adata.var["highly_variable"].astype(float)))
    print(f"[OK] HVGs computed upfront: n={n_hvg}")


if RUN_PANEL_A_LLM or RUN_PANEL_C_LLM:
    adata = sc.read_h5ad(DATA_PATH)
    assert "meta.cluster" in adata.obs.columns, "meta.cluster not found in adata.obs"

    adata.obs["Ground_Truth"] = adata.obs["meta.cluster"].astype(str).apply(get_cd8_ground_truth)

    print(adata)
    print("\nGround truth counts:")
    print(adata.obs["Ground_Truth"].value_counts(dropna=False))

    ensure_hvgs_for_cd8(adata)

    try:
        curator.set_global_context(adata)
        print("\n[OK] curator global context set.")
    except Exception as e:
        print(f"\n[WARN] curator.set_global_context(adata) failed: {e}")
        print("Proceeding, but feature distillation may be less stable if global context is unset.")

AnnData object with n_obs × n_vars = 4466 × 24148
    obs: 'cancerType', 'patient', 'libraryID', 'loc', 'meta.cluster', 'platform', 'Cancer_Type', 'Sample_ID', 'Ground_Truth'
    uns: 'log1p'
    layers: 'counts', 'logcounts'

Ground truth counts:
Ground_Truth
CD8_EffectorMemory    1983
CD8_Exhausted          928
CD8_Effector           753
CD8_Naive              300
CD8_MAIT               300
CD8_ISG                202
Name: count, dtype: int64
[INFO] highly_variable not found. Computing HVGs once upfront...
[OK] HVGs computed upfront: n=2000

[OK] curator global context set.


In [57]:
# =============================================================================
# 5. Helper functions
# =============================================================================
def select_cluster_meta(
    adata,
    exclude_gt_labels=None,
    focus_gt_labels=None,
    max_clusters_per_gt=None,
):
    if exclude_gt_labels is None:
        exclude_gt_labels = set()

    unique_clusters = sorted(adata.obs["meta.cluster"].astype(str).unique())
    cluster_meta_all = []

    for c in unique_clusters:
        gt = get_cd8_ground_truth(c)
        if gt in exclude_gt_labels or gt is None:
            continue
        if focus_gt_labels is not None and gt not in set(focus_gt_labels):
            continue
        cluster_meta_all.append((c, gt))

    if max_clusters_per_gt is None:
        return cluster_meta_all

    selected = []
    counts = defaultdict(int)
    for c, gt in cluster_meta_all:
        if counts[gt] >= max_clusters_per_gt:
            continue
        selected.append((c, gt))
        counts[gt] += 1

    return selected


def safe_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def maybe_to_percent(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype(float)
    if s.dropna().empty:
        return s
    if s.max() <= 1.05:
        return s * 100.0
    return s


def summarize_with_ci(df: pd.DataFrame, group_cols, value_col: str) -> pd.DataFrame:
    out = (
        df.groupby(group_cols, as_index=False)
        .agg(
            mean=(value_col, "mean"),
            std=(value_col, "std"),
            n=(value_col, "count"),
        )
    )
    out["std"] = out["std"].fillna(0.0)
    out["se"] = out["std"] / np.sqrt(out["n"].clip(lower=1))
    z = 1.96
    out["lower"] = out["mean"] - z * out["se"]
    out["upper"] = out["mean"] + z * out["se"]
    return out


def choose_first_existing(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def prediction_text_for_scoring(answer: str, reasoning: str, mode: str = "cell_type_primary") -> str:
    a = safe_text(answer)
    r = safe_text(reasoning)

    if mode == "cell_type_plus_reasoning":
        if a and r:
            return f"{a} {r}"
        return a if a else r

    # conservative default
    return a if a else r


def score_cd8_answer_text(gt_label: str, pred_text: str) -> float:
    row = pd.Series(
        {
            "Ground_Truth": safe_text(gt_label),
            "Pred_Text": safe_text(pred_text),
        }
    )
    return float(score_hierarchical(row, "Pred_Text", cfg=CD8_HIER_CFG))


def normalize_annotation_result(x):
    if isinstance(x, dict):
        return {
            "cell_type": str(x.get("cell_type", "")),
            "confidence": str(x.get("confidence", "")),
            "reasoning": str(x.get("reasoning", "")),
        }
    return {
        "cell_type": str(x),
        "confidence": "",
        "reasoning": "",
    }


def cache_key(payload: dict) -> str:
    s = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def annotate_with_cache(
    curator,
    payload: dict,
    sleep_sec: float = 0.0,
):
    key = cache_key(payload)
    cache_path = CACHE_DIR / f"{key}.json"

    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    raw = curator.annotate(
        payload["genes"],
        use_auto_context=payload["use_auto_context"],
    )
    result = normalize_annotation_result(raw)

    out = {
        "payload": payload,
        "result": result,
    }
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    if sleep_sec > 0:
        time.sleep(sleep_sec)

    return out


def build_noise_pool_from_package(adata, min_pool_size: int = 50) -> list:
    all_genes = adata.var_names.astype(str).tolist()
    noise_set = set()

    for pattern in NOISE_PATTERNS.values():
        regex = re.compile(pattern)
        matches = [g for g in all_genes if regex.search(g)]
        noise_set.update(matches)

    for gene_set in NOISE_LISTS.values():
        matches = [g for g in all_genes if g in gene_set]
        noise_set.update(matches)

    pool = list(noise_set)

    if len(pool) < min_pool_size:
        defaults = list(CELL_CYCLE_GENES) + [
            "ACTB", "GAPDH", "MALAT1", "NEAT1", "RPS12", "MT-CO1", "JUN", "FOS"
        ]
        pool = list(set(pool).union(defaults))

    print(f"[INFO] Noise pool size: {len(pool)}")
    print(f"[INFO] Noise pool examples: {pool[:10]}")
    return pool


def build_true_marker_pool(
    adata,
    cluster_name: str,
    noise_pool: list,
    group_col: str = "meta.cluster",
    n_candidates: int = 300,
) -> list:
    adata_temp = adata.copy()
    adata_temp.obs["binary_group"] = "Rest"
    adata_temp.obs.loc[
        adata_temp.obs[group_col].astype(str) == str(cluster_name), "binary_group"
    ] = "Target"

    try:
        sc.tl.rank_genes_groups(
            adata_temp,
            groupby="binary_group",
            groups=["Target"],
            reference="Rest",
            method="wilcoxon",
            use_raw=False,
        )
        df_de = sc.get.rank_genes_groups_df(adata_temp, group="Target")
        raw_signal = df_de["names"].astype(str).tolist()
        true_markers = [g for g in raw_signal if g not in set(noise_pool)][:n_candidates]
    except Exception as e:
        print(f"[WARN] rank_genes_groups failed for {cluster_name}: {e}")
        true_markers = [g for g in adata.var_names.astype(str).tolist() if g not in set(noise_pool)][:n_candidates]

    del adata_temp
    return true_markers


In [58]:
# =============================================================================
# 6. Cluster selection
# =============================================================================
cluster_meta_a = None
cluster_meta_c = None

if adata is not None:
    cluster_meta_a = select_cluster_meta(
        adata=adata,
        exclude_gt_labels=EXCLUDE_GT_LABELS,
        focus_gt_labels=PANEL_A_FOCUS_GT_LABELS,
        max_clusters_per_gt=PANEL_A_MAX_CLUSTERS_PER_GT,
    )

    cluster_meta_c = select_cluster_meta(
        adata=adata,
        exclude_gt_labels=EXCLUDE_GT_LABELS,
        focus_gt_labels=PANEL_C_FOCUS_GT_LABELS,
        max_clusters_per_gt=PANEL_C_MAX_CLUSTERS_PER_GT,
    )

    print("\n[Panel a/b] Selected clusters:")
    for c, gt in cluster_meta_a:
        print(f"  - {c} (GT={gt})")
    print(f"[Panel a/b] n_clusters = {len(cluster_meta_a)}")

    print("\n[Panel c] Selected clusters:")
    for c, gt in cluster_meta_c:
        print(f"  - {c} (GT={gt})")
    print(f"[Panel c] n_clusters = {len(cluster_meta_c)}")

    pd.DataFrame(cluster_meta_a, columns=["Cluster_ID", "Ground_Truth"]).to_csv(
        OUTPUT_DIR / "Fig4_selected_clusters_panel_ab.csv", index=False
    )
    pd.DataFrame(cluster_meta_c, columns=["Cluster_ID", "Ground_Truth"]).to_csv(
        OUTPUT_DIR / "Fig4_selected_clusters_panel_c.csv", index=False
    )


[Panel a/b] Selected clusters:
  - CD8.c01.Tn.MAL (GT=CD8_Naive)
  - CD8.c02.Tm.IL7R (GT=CD8_EffectorMemory)
  - CD8.c03.Tm.RPS12 (GT=CD8_EffectorMemory)
  - CD8.c04.Tm.CD52 (GT=CD8_EffectorMemory)
  - CD8.c05.Tem.CXCR5 (GT=CD8_EffectorMemory)
  - CD8.c06.Tem.GZMK (GT=CD8_EffectorMemory)
  - CD8.c07.Temra.CX3CR1 (GT=CD8_Effector)
  - CD8.c08.Tk.TYROBP (GT=CD8_Effector)
  - CD8.c09.Tk.KIR2DL4 (GT=CD8_Effector)
  - CD8.c10.Trm.ZNF683 (GT=CD8_EffectorMemory)
  - CD8.c11.Tex.PDCD1 (GT=CD8_Exhausted)
  - CD8.c12.Tex.CXCL13 (GT=CD8_Exhausted)
  - CD8.c13.Tex.myl12a (GT=CD8_Exhausted)
  - CD8.c14.Tex.TCF7 (GT=CD8_Exhausted)
  - CD8.c15.ISG.IFIT1 (GT=CD8_ISG)
  - CD8.c16.MAIT.SLC4A10 (GT=CD8_MAIT)
  - CD8.c17.Tm.NME1 (GT=CD8_EffectorMemory)
[Panel a/b] n_clusters = 17

[Panel c] Selected clusters:
  - CD8.c01.Tn.MAL (GT=CD8_Naive)
  - CD8.c02.Tm.IL7R (GT=CD8_EffectorMemory)
  - CD8.c03.Tm.RPS12 (GT=CD8_EffectorMemory)
  - CD8.c07.Temra.CX3CR1 (GT=CD8_Effector)
  - CD8.c08.Tk.TYROBP (GT=CD8_Ef

In [59]:
# =============================================================================
# 7. Panel A LLM generation (saved raw outputs)
# =============================================================================
def generate_panel_a_raw_outputs(
    adata,
    curator,
    cluster_meta,
    n_genes_list,
    out_csv,
    group_col="meta.cluster",
    force_rerun=False,
):
    if out_csv.exists() and not force_rerun:
        print(f"[INFO] Loading existing panel A raw outputs: {out_csv}")
        return pd.read_csv(out_csv)

    rank_cache = {}
    curated_cache = {}

    print("\n[Panel A] Computing Standard DE ranks once per cluster...")
    for i, (cluster_name, gt_label) in enumerate(cluster_meta, start=1):
        print(f"  [STD {i}/{len(cluster_meta)}] {cluster_name} -> {gt_label}")

        adata.obs["binary_group"] = "Rest"
        adata.obs.loc[
            adata.obs[group_col].astype(str) == str(cluster_name),
            "binary_group"
        ] = "Target"

        try:
            sc.tl.rank_genes_groups(
                adata,
                groupby="binary_group",
                groups=["Target"],
                reference="Rest",
                method="wilcoxon",
                use_raw=False,
            )
            df_rank = sc.get.rank_genes_groups_df(adata, group="Target")
            rank_cache[cluster_name] = df_rank["names"].astype(str).tolist()
        except Exception as e:
            print(f"  [WARN] rank_genes_groups failed for {cluster_name}: {e}")
            rank_cache[cluster_name] = []

    adata.obs.drop(columns=["binary_group"], inplace=True, errors="ignore")

    print("\n[Panel A] Computing Curated gene lists per cluster and per N...")
    for i, (cluster_name, gt_label) in enumerate(cluster_meta, start=1):
        print(f"  [CUR {i}/{len(cluster_meta)}] {cluster_name} -> {gt_label}")
        for n_genes in n_genes_list:
            key = (str(cluster_name), int(n_genes))
            try:
                genes_cur = curator.curate_features(
                    adata,
                    group_col=group_col,
                    target_group=cluster_name,
                    n_top=n_genes,
                    use_statistics=True,
                )
                curated_cache[key] = [str(g) for g in genes_cur]
            except Exception as e:
                print(f"  [WARN] curate_features failed for {cluster_name}, N={n_genes}: {e}")
                curated_cache[key] = []

    rows = []
    print("\n[Panel A] Generating cached LLM outputs...")

    for n_genes in n_genes_list:
        print(f"\n--- Top {n_genes} genes ---")

        for cluster_name, gt_label in cluster_meta:
            genes_std = rank_cache.get(cluster_name, [])[:n_genes]
            genes_cur = curated_cache.get((str(cluster_name), int(n_genes)), [])

            print(f"  ▶ {cluster_name} (GT={gt_label}), N={n_genes}")

            payload_std = {
                "panel": "A",
                "method": "Standard",
                "dataset": "CD8",
                "cluster_id": str(cluster_name),
                "ground_truth": str(gt_label),
                "n_genes": int(n_genes),
                "use_auto_context": bool(STANDARD_USE_AUTO_CONTEXT_FOR_FIG4),
                "genes": list(genes_std),
                "model_name": MODEL_NAME,
            }
            out_std = annotate_with_cache(curator, payload_std, sleep_sec=SLEEP_STD)
            res_std = out_std["result"]
            std_pred_text = prediction_text_for_scoring(
                res_std.get("cell_type", ""),
                res_std.get("reasoning", ""),
                mode=SCORING_TEXT_MODE,
            )

            payload_cur = {
                "panel": "A",
                "method": "LLM-scCurator",
                "dataset": "CD8",
                "cluster_id": str(cluster_name),
                "ground_truth": str(gt_label),
                "n_genes": int(n_genes),
                "use_auto_context": bool(CURATOR_USE_AUTO_CONTEXT_FOR_FIG4),
                "genes": list(genes_cur),
                "model_name": MODEL_NAME,
            }
            out_cur = annotate_with_cache(curator, payload_cur, sleep_sec=SLEEP_CUR)
            res_cur = out_cur["result"]
            cur_pred_text = prediction_text_for_scoring(
                res_cur.get("cell_type", ""),
                res_cur.get("reasoning", ""),
                mode=SCORING_TEXT_MODE,
            )

            rows.append(
                {
                    "Cluster_ID": str(cluster_name),
                    "Ground_Truth": str(gt_label),
                    "N_Genes": int(n_genes),

                    "Std_UseAutoContext": bool(STANDARD_USE_AUTO_CONTEXT_FOR_FIG4),
                    "Std_Answer": safe_text(res_std.get("cell_type", "")),
                    "Std_Confidence": safe_text(res_std.get("confidence", "")),
                    "Std_Reasoning": safe_text(res_std.get("reasoning", "")),
                    "Std_PredText_ForScoring": std_pred_text,
                    "Std_Genes": "|".join(genes_std),

                    "Cur_UseAutoContext": bool(CURATOR_USE_AUTO_CONTEXT_FOR_FIG4),
                    "Cur_Answer": safe_text(res_cur.get("cell_type", "")),
                    "Cur_Confidence": safe_text(res_cur.get("confidence", "")),
                    "Cur_Reasoning": safe_text(res_cur.get("reasoning", "")),
                    "Cur_PredText_ForScoring": cur_pred_text,
                    "Cur_Genes": "|".join(genes_cur),

                    "Model_Name": MODEL_NAME,
                    "Scoring_Text_Mode": SCORING_TEXT_MODE,
                }
            )

        pd.DataFrame(rows).to_csv(
            OUTPUT_DIR / "panelA_input_size_intermediate_raw_outputs.csv",
            index=False,
        )

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"[OK] Panel A raw outputs saved -> {out_csv}")
    return df

In [60]:
# =============================================================================
# 8. Panel C LLM generation (saved raw outputs)
# =============================================================================
def generate_panel_c_raw_outputs(
    adata,
    curator,
    cluster_meta,
    out_csv,
    noise_ratios=(0.0, 0.2, 0.4, 0.6, 0.8),
    n_markers=50,
    n_repeats=5,
    random_seed=42,
    group_col="meta.cluster",
    force_rerun=False,
):
    if out_csv.exists() and not force_rerun:
        print(f"[INFO] Loading existing panel C raw outputs: {out_csv}")
        return pd.read_csv(out_csv)

    rng = np.random.default_rng(random_seed)
    noise_pool = build_noise_pool_from_package(adata)
    noise_pool_set = set(noise_pool)

    distiller = FeatureDistiller(adata)
    mask_reasons_global = distiller.detect_biological_noise(
        gini_threshold=None,
        gini_q=0.01,
        mean_floor=0.01,
    )
    mask_reasons_global = set(mask_reasons_global.keys())

    print(f"\n[Panel C] Global masked genes: {len(mask_reasons_global)}")

    rows = []

    for i, (cluster_name, gt_label) in enumerate(cluster_meta, start=1):
        print(f"\n[Panel C] Cluster [{i}/{len(cluster_meta)}]: {cluster_name} (GT={gt_label})")

        true_markers_pool = build_true_marker_pool(
            adata=adata,
            cluster_name=cluster_name,
            noise_pool=noise_pool,
            group_col=group_col,
            n_candidates=300,
        )

        for ratio in noise_ratios:
            pct = int(ratio * 100)
            n_noise = int(n_markers * ratio)
            n_signal = n_markers - n_noise

            print(f"  --- Noise ratio: {pct}% ---")

            for rep in range(n_repeats):
                if n_noise > 0 and len(noise_pool) > 0:
                    idx_noise = rng.choice(
                        len(noise_pool),
                        size=n_noise,
                        replace=(len(noise_pool) < n_noise),
                    )
                    current_noise = [noise_pool[j] for j in idx_noise]
                else:
                    current_noise = []

                if n_signal > 0 and len(true_markers_pool) > 0:
                    idx_signal = rng.choice(
                        len(true_markers_pool),
                        size=n_signal,
                        replace=(len(true_markers_pool) < n_signal),
                    )
                    current_signal = [true_markers_pool[j] for j in idx_signal]
                else:
                    current_signal = []

                genes_std = current_noise + current_signal
                rng.shuffle(genes_std)

                genes_cur_filtered = [g for g in genes_std if g not in mask_reasons_global]
                genes_cur = genes_cur_filtered[:]

                if len(genes_cur) < n_markers:
                    needed = n_markers - len(genes_cur)
                    refill_candidates = [g for g in true_markers_pool if g not in genes_cur]

                    if needed > len(refill_candidates) and len(true_markers_pool) > 0:
                        extra_needed = needed - len(refill_candidates)
                        extra_idx = rng.choice(
                            len(true_markers_pool),
                            size=extra_needed,
                            replace=True,
                        )
                        refill_candidates.extend([true_markers_pool[j] for j in extra_idx])

                    genes_cur.extend(refill_candidates[:needed])

                genes_cur = genes_cur[:n_markers]

                noise_in_std = [g for g in genes_std if g in noise_pool_set]
                noise_in_cur = [g for g in genes_cur if g in noise_pool_set]

                payload_std = {
                    "panel": "C",
                    "method": "Standard",
                    "dataset": "CD8",
                    "cluster_id": str(cluster_name),
                    "ground_truth": str(gt_label),
                    "noise_ratio": float(ratio),
                    "repeat_id": int(rep),
                    "n_genes": int(n_markers),
                    "use_auto_context": bool(STANDARD_USE_AUTO_CONTEXT_FOR_FIG4),
                    "genes": list(genes_std),
                    "model_name": MODEL_NAME,
                }
                out_std = annotate_with_cache(curator, payload_std, sleep_sec=SLEEP_STD)
                res_std = out_std["result"]
                std_pred_text = prediction_text_for_scoring(
                    res_std.get("cell_type", ""),
                    res_std.get("reasoning", ""),
                    mode=SCORING_TEXT_MODE,
                )

                payload_cur = {
                    "panel": "C",
                    "method": "LLM-scCurator",
                    "dataset": "CD8",
                    "cluster_id": str(cluster_name),
                    "ground_truth": str(gt_label),
                    "noise_ratio": float(ratio),
                    "repeat_id": int(rep),
                    "n_genes": int(n_markers),
                    "use_auto_context": bool(CURATOR_USE_AUTO_CONTEXT_FOR_FIG4),
                    "genes": list(genes_cur),
                    "model_name": MODEL_NAME,
                }
                out_cur = annotate_with_cache(curator, payload_cur, sleep_sec=SLEEP_CUR)
                res_cur = out_cur["result"]
                cur_pred_text = prediction_text_for_scoring(
                    res_cur.get("cell_type", ""),
                    res_cur.get("reasoning", ""),
                    mode=SCORING_TEXT_MODE,
                )

                rows.append(
                    {
                        "Cluster_ID": str(cluster_name),
                        "Ground_Truth": str(gt_label),
                        "Repeat_ID": int(rep),
                        "Noise_Ratio": float(ratio),
                        "Noise_Pct": float(ratio) * 100.0,
                        "N_Genes": int(n_markers),

                        "N_Noise_Std": int(len(noise_in_std)),
                        "N_Noise_Cur": int(len(noise_in_cur)),

                        "Std_UseAutoContext": bool(STANDARD_USE_AUTO_CONTEXT_FOR_FIG4),
                        "Std_Answer": safe_text(res_std.get("cell_type", "")),
                        "Std_Confidence": safe_text(res_std.get("confidence", "")),
                        "Std_Reasoning": safe_text(res_std.get("reasoning", "")),
                        "Std_PredText_ForScoring": std_pred_text,
                        "Std_Genes": "|".join(genes_std),

                        "Cur_UseAutoContext": bool(CURATOR_USE_AUTO_CONTEXT_FOR_FIG4),
                        "Cur_Answer": safe_text(res_cur.get("cell_type", "")),
                        "Cur_Confidence": safe_text(res_cur.get("confidence", "")),
                        "Cur_Reasoning": safe_text(res_cur.get("reasoning", "")),
                        "Cur_PredText_ForScoring": cur_pred_text,
                        "Cur_Genes": "|".join(genes_cur),

                        "Model_Name": MODEL_NAME,
                        "Scoring_Text_Mode": SCORING_TEXT_MODE,
                    }
                )

                print(
                    f"    rep {rep+1}/{n_repeats} | "
                    f"noise std={len(noise_in_std)} cur={len(noise_in_cur)}"
                )

                time.sleep(SLEEP_BETWEEN_REPEATS)

        pd.DataFrame(rows).to_csv(
            OUTPUT_DIR / "Fig4c.csv",
            index=False,
        )

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"[OK] Panel C raw outputs saved -> {out_csv}")
    return df



In [61]:
# =============================================================================
# 9. Reevaluation from saved answers
# =============================================================================
def reevaluate_panel_a_from_saved_answers(input_csv: Path, output_csv: Path) -> pd.DataFrame:
    if not input_csv.exists():
        raise FileNotFoundError(f"Missing panel A raw output CSV: {input_csv}")

    df = pd.read_csv(input_csv)

    if "Ground_Truth" not in df.columns and "Cluster_ID" in df.columns:
        df["Ground_Truth"] = df["Cluster_ID"].astype(str).apply(get_cd8_ground_truth)

    if "Std_PredText_ForScoring" in df.columns:
        df["Std_PredText_ForScoring"] = df["Std_PredText_ForScoring"].fillna("")
    else:
        answer_col = choose_first_existing(df, ["Std_Answer", "Standard_Answer"])
        reasoning_col = choose_first_existing(df, ["Std_Reasoning", "Standard_Reasoning"])
        df["Std_PredText_ForScoring"] = df.apply(
            lambda r: prediction_text_for_scoring(
                r[answer_col] if answer_col else "",
                r[reasoning_col] if reasoning_col else "",
                mode=SCORING_TEXT_MODE,
            ),
            axis=1,
        )

    if "Cur_PredText_ForScoring" in df.columns:
        df["Cur_PredText_ForScoring"] = df["Cur_PredText_ForScoring"].fillna("")
    else:
        answer_col = choose_first_existing(df, ["Cur_Answer", "Curator_Answer", "Curated_Answer"])
        reasoning_col = choose_first_existing(df, ["Cur_Reasoning", "Curator_Reasoning", "Curated_Reasoning"])
        df["Cur_PredText_ForScoring"] = df.apply(
            lambda r: prediction_text_for_scoring(
                r[answer_col] if answer_col else "",
                r[reasoning_col] if reasoning_col else "",
                mode=SCORING_TEXT_MODE,
            ),
            axis=1,
        )

    if "Std_Score" in df.columns:
        df["Std_Score_OLD"] = pd.to_numeric(df["Std_Score"], errors="coerce")
    if "Cur_Score" in df.columns:
        df["Cur_Score_OLD"] = pd.to_numeric(df["Cur_Score"], errors="coerce")

    df["Std_Score"] = df.apply(
        lambda r: score_cd8_answer_text(r["Ground_Truth"], r["Std_PredText_ForScoring"]),
        axis=1,
    )
    df["Cur_Score"] = df.apply(
        lambda r: score_cd8_answer_text(r["Ground_Truth"], r["Cur_PredText_ForScoring"]),
        axis=1,
    )
    df["Score_Diff"] = df["Cur_Score"] - df["Std_Score"]

    df.to_csv(output_csv, index=False)
    print(f"[OK] Re-evaluated panel A saved -> {output_csv}")
    return df


def reevaluate_panel_c_from_saved_answers(input_csv: Path, output_csv: Path) -> pd.DataFrame:
    if not input_csv.exists():
        raise FileNotFoundError(f"Missing panel C raw output CSV: {input_csv}")

    df = pd.read_csv(input_csv)

    if "Ground_Truth" not in df.columns and "Cluster_ID" in df.columns:
        df["Ground_Truth"] = df["Cluster_ID"].astype(str).apply(get_cd8_ground_truth)

    if "Std_PredText_ForScoring" in df.columns:
        df["Std_PredText_ForScoring"] = df["Std_PredText_ForScoring"].fillna("")
    else:
        answer_col = choose_first_existing(df, ["Std_Answer", "Standard_Answer"])
        reasoning_col = choose_first_existing(df, ["Std_Reasoning", "Standard_Reasoning"])
        df["Std_PredText_ForScoring"] = df.apply(
            lambda r: prediction_text_for_scoring(
                r[answer_col] if answer_col else "",
                r[reasoning_col] if reasoning_col else "",
                mode=SCORING_TEXT_MODE,
            ),
            axis=1,
        )

    if "Cur_PredText_ForScoring" in df.columns:
        df["Cur_PredText_ForScoring"] = df["Cur_PredText_ForScoring"].fillna("")
    else:
        answer_col = choose_first_existing(df, ["Cur_Answer", "Curator_Answer", "Curated_Answer"])
        reasoning_col = choose_first_existing(df, ["Cur_Reasoning", "Curator_Reasoning", "Curated_Reasoning"])
        df["Cur_PredText_ForScoring"] = df.apply(
            lambda r: prediction_text_for_scoring(
                r[answer_col] if answer_col else "",
                r[reasoning_col] if reasoning_col else "",
                mode=SCORING_TEXT_MODE,
            ),
            axis=1,
        )

    if "Standard_Acc" in df.columns:
        df["Standard_Acc_OLD"] = pd.to_numeric(df["Standard_Acc"], errors="coerce")
    if "Curator_Acc" in df.columns:
        df["Curator_Acc_OLD"] = pd.to_numeric(df["Curator_Acc"], errors="coerce")

    df["Standard_Acc"] = df.apply(
        lambda r: score_cd8_answer_text(r["Ground_Truth"], r["Std_PredText_ForScoring"]),
        axis=1,
    )
    df["Curator_Acc"] = df.apply(
        lambda r: score_cd8_answer_text(r["Ground_Truth"], r["Cur_PredText_ForScoring"]),
        axis=1,
    )
    df["Was_Reevaluated"] = True

    df.to_csv(output_csv, index=False)
    print(f"[OK] Re-evaluated panel C saved -> {output_csv}")
    return df


In [13]:
# =============================================================================
# 10. Optional generation step
# =============================================================================
if RUN_PANEL_A_LLM:
    df_panel_a_raw = generate_panel_a_raw_outputs(
        adata=adata,
        curator=curator,
        cluster_meta=cluster_meta_a,
        n_genes_list=N_GENES_LIST,
        out_csv=PANEL_A_RAW_CSV,
        force_rerun=FORCE_RERUN_PANEL_A,
    )


[Panel A] Computing Standard DE ranks once per cluster...
  [STD 1/17] CD8.c01.Tn.MAL -> CD8_Naive
  [STD 2/17] CD8.c02.Tm.IL7R -> CD8_EffectorMemory
  [STD 3/17] CD8.c03.Tm.RPS12 -> CD8_EffectorMemory
  [STD 4/17] CD8.c04.Tm.CD52 -> CD8_EffectorMemory
  [STD 5/17] CD8.c05.Tem.CXCR5 -> CD8_EffectorMemory
  [STD 6/17] CD8.c06.Tem.GZMK -> CD8_EffectorMemory
  [STD 7/17] CD8.c07.Temra.CX3CR1 -> CD8_Effector
  [STD 8/17] CD8.c08.Tk.TYROBP -> CD8_Effector
  [STD 9/17] CD8.c09.Tk.KIR2DL4 -> CD8_Effector
  [STD 10/17] CD8.c10.Trm.ZNF683 -> CD8_EffectorMemory
  [STD 11/17] CD8.c11.Tex.PDCD1 -> CD8_Exhausted
  [STD 12/17] CD8.c12.Tex.CXCL13 -> CD8_Exhausted
  [STD 13/17] CD8.c13.Tex.myl12a -> CD8_Exhausted
  [STD 14/17] CD8.c14.Tex.TCF7 -> CD8_Exhausted
  [STD 15/17] CD8.c15.ISG.IFIT1 -> CD8_ISG
  [STD 16/17] CD8.c16.MAIT.SLC4A10 -> CD8_MAIT
  [STD 17/17] CD8.c17.Tm.NME1 -> CD8_EffectorMemory

[Panel A] Computing Curated gene lists per cluster and per N...
  [CUR 1/17] CD8.c01.Tn.MAL -> CD8_Na

KeyboardInterrupt: 

In [68]:
def delete_error_cache_only(cache_dir):
    cache_dir = Path(cache_dir)
    deleted = []
    kept = []
    failed_to_read = []

    for fp in cache_dir.glob("*.json"):
        try:
            with open(fp, "r", encoding="utf-8") as f:
                obj = json.load(f)

            status = str(obj.get("status", "")).strip().lower()
            error_message = str(obj.get("error_message", "")).strip()

            result = obj.get("result", {}) or {}
            cell_type = str(result.get("cell_type", "")).strip()
            reasoning = str(result.get("reasoning", "")).strip()

            is_error = (
                status == "error"
                or cell_type == "Error"
                or "quota exceeded" in error_message.lower()
                or "quota exceeded" in reasoning.lower()
                or "429" in error_message
                or "429" in reasoning
            )

            if is_error:
                fp.unlink()
                deleted.append(fp.name)
            else:
                kept.append(fp.name)

        except Exception as e:
            failed_to_read.append((fp.name, str(e)))

    print(f"[OK] Deleted error cache files: {len(deleted)}")
    print(f"[OK] Kept non-error cache files: {len(kept)}")

    if deleted:
        print("\nDeleted examples:")
        for x in deleted[:10]:
            print(" -", x)

    if failed_to_read:
        print("\n[WARN] Failed to read some cache files:")
        for name, msg in failed_to_read[:10]:
            print(f" - {name}: {msg}")

    return deleted, kept, failed_to_read

def remove_error_rows_from_panel_raw_csv(raw_csv_path, backup=True):
    raw_csv_path = Path(raw_csv_path)

    if not raw_csv_path.exists():
        raise FileNotFoundError(f"Raw CSV not found: {raw_csv_path}")

    df = pd.read_csv(raw_csv_path)
    df_before = df.copy()

    for col in ["Std_Answer", "Cur_Answer", "Std_Reasoning", "Cur_Reasoning"]:
        if col not in df.columns:
            df[col] = ""

    mask_error = (
        (df["Std_Answer"].astype(str).str.strip() == "Error")
        | (df["Cur_Answer"].astype(str).str.strip() == "Error")
        | (df["Std_Reasoning"].astype(str).str.contains("quota exceeded|429", case=False, na=False))
        | (df["Cur_Reasoning"].astype(str).str.contains("quota exceeded|429", case=False, na=False))
    )

    n_error = int(mask_error.sum())
    n_keep = int((~mask_error).sum())

    if backup:
        backup_path = raw_csv_path.with_suffix(raw_csv_path.suffix + ".backup_before_error_row_removal.csv")
        df_before.to_csv(backup_path, index=False)
        print(f"[OK] Backup saved -> {backup_path}")

    df_clean = df.loc[~mask_error].copy()
    df_clean.to_csv(raw_csv_path, index=False)

    print(f"[OK] Removed error rows: {n_error}")
    print(f"[OK] Remaining rows    : {n_keep}")
    print(f"[OK] Cleaned raw CSV   : {raw_csv_path}")

    return df_clean

In [62]:
if RUN_PANEL_C_LLM:
    df_panel_c_raw = generate_panel_c_raw_outputs(
        adata=adata,
        curator=curator,
        cluster_meta=cluster_meta_c,
        out_csv=PANEL_C_RAW_CSV,
        noise_ratios=NOISE_RATIOS,
        n_markers=N_MARKERS_NOISE,
        n_repeats=N_REPEATS_NOISE,
        random_seed=RANDOM_SEED,
        force_rerun=FORCE_RERUN_PANEL_C,
    )

[INFO] Noise pool size: 1235
[INFO] Noise pool examples: ['IGKV2D-24', 'HLA-G', 'HIST1H2BN', 'TRBJ1-2', 'LINC00240', 'HSPB11', 'LINC00601', 'HIST2H2AA4', 'H2AFZ', 'LINC01354']

[Panel C] Global masked genes: 1255

[Panel C] Cluster [1/9]: CD8.c01.Tn.MAL (GT=CD8_Naive)
  --- Noise ratio: 0% ---
    rep 1/5 | noise std=0 cur=0
    rep 2/5 | noise std=0 cur=0
    rep 3/5 | noise std=0 cur=0
    rep 4/5 | noise std=0 cur=0
    rep 5/5 | noise std=0 cur=0
  --- Noise ratio: 20% ---
    rep 1/5 | noise std=10 cur=0
    rep 2/5 | noise std=10 cur=0
    rep 3/5 | noise std=10 cur=0
    rep 4/5 | noise std=10 cur=0
    rep 5/5 | noise std=10 cur=0
  --- Noise ratio: 40% ---
    rep 1/5 | noise std=20 cur=0
    rep 2/5 | noise std=20 cur=0
    rep 3/5 | noise std=20 cur=0
    rep 4/5 | noise std=20 cur=0
    rep 5/5 | noise std=20 cur=0
  --- Noise ratio: 60% ---
    rep 1/5 | noise std=30 cur=0
    rep 2/5 | noise std=30 cur=0
    rep 3/5 | noise std=30 cur=0
    rep 4/5 | noise std=30 cur=0
  

In [69]:
df = pd.read_csv(PANEL_C_RAW_CSV)

mask_error = (
    (df["Std_Answer"].astype(str).str.strip() == "Error")
    | (df["Cur_Answer"].astype(str).str.strip() == "Error")
    | (df["Std_Reasoning"].astype(str).str.contains("quota exceeded|429", case=False, na=False))
    | (df["Cur_Reasoning"].astype(str).str.contains("quota exceeded|429", case=False, na=False))
)

print("Total rows :", len(df))
print("Error rows :", int(mask_error.sum()))
print("Clean rows :", int((~mask_error).sum()))

Total rows : 225
Error rows : 67
Clean rows : 158


In [70]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

panel_c_backup = PANEL_C_RAW_CSV.with_name(f"{PANEL_C_RAW_CSV.stem}.backup_{ts}.csv")
shutil.copy2(PANEL_C_RAW_CSV, panel_c_backup)

cache_backup_dir = CACHE_DIR.parent / f"{CACHE_DIR.name}_backup_{ts}"
shutil.copytree(CACHE_DIR, cache_backup_dir)

print("Backed up raw CSV ->", panel_c_backup)
print("Backed up cache dir ->", cache_backup_dir)

Backed up raw CSV -> /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelC_multicluster_noise_simulation_raw_outputs.backup_20260314_152919.csv
Backed up cache dir -> /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/llm_cache_backup_20260314_152919


In [72]:
deleted, kept, failed = delete_error_cache_only(CACHE_DIR)

[OK] Deleted error cache files: 0
[OK] Kept non-error cache files: 486


In [73]:
test_genes = ["IL7R", "TCF7", "CCR7", "LTB", "MAL", "LEF1", "KLF2", "SELL"]

t0 = time.time()
res = curator.annotate(test_genes, use_auto_context=True)
dt = time.time() - t0

print(f"Elapsed: {dt:.2f} sec")
print(json.dumps(res, ensure_ascii=False, indent=2) if isinstance(res, dict) else str(res))

Elapsed: 9.34 sec
{
  "cell_type": "Naive T cell",
  "confidence": "High",
  "reasoning": "The gene set includes canonical markers for naive T cells, including the transcription factors TCF7, LEF1, and KLF2, and the lymph node homing receptors CCR7 and SELL (CD62L). IL7R is also highly expressed for homeostatic maintenance."
}


In [75]:
def inspect_panel_c_cache_plan(
    adata,
    cluster_meta,
    cache_dir,
    noise_ratios=(0.0, 0.2, 0.4, 0.6, 0.8),
    n_markers=50,
    n_repeats=5,
    random_seed=42,
    group_col="meta.cluster",
):
    """
    generate_panel_c_raw_outputs() と同じ payload を組み立て、
    APIを呼ばずに cache hit / miss を事前確認する。
    """

    rng = np.random.default_rng(random_seed)
    cache_dir = Path(cache_dir)

    noise_pool = build_noise_pool_from_package(adata)
    noise_pool_set = set(noise_pool)

    distiller = FeatureDistiller(adata)
    mask_reasons_global = distiller.detect_biological_noise(
        gini_threshold=None,
        gini_q=0.01,
        mean_floor=0.01,
    )
    mask_reasons_global = set(mask_reasons_global.keys())

    print(f"[CHECK] Global masked genes: {len(mask_reasons_global)}")

    rows = []

    for i, (cluster_name, gt_label) in enumerate(cluster_meta, start=1):
        print(f"\n[CHECK] Cluster [{i}/{len(cluster_meta)}]: {cluster_name} (GT={gt_label})")

        true_markers_pool = build_true_marker_pool(
            adata=adata,
            cluster_name=cluster_name,
            noise_pool=noise_pool,
            group_col=group_col,
            n_candidates=300,
        )

        for ratio in noise_ratios:
            pct = int(ratio * 100)
            n_noise = int(n_markers * ratio)
            n_signal = n_markers - n_noise

            print(f"  --- Noise ratio: {pct}% ---")

            for rep in range(n_repeats):
                # Standard gene list
                if n_noise > 0 and len(noise_pool) > 0:
                    idx_noise = rng.choice(
                        len(noise_pool),
                        size=n_noise,
                        replace=(len(noise_pool) < n_noise),
                    )
                    current_noise = [noise_pool[j] for j in idx_noise]
                else:
                    current_noise = []

                if n_signal > 0 and len(true_markers_pool) > 0:
                    idx_signal = rng.choice(
                        len(true_markers_pool),
                        size=n_signal,
                        replace=(len(true_markers_pool) < n_signal),
                    )
                    current_signal = [true_markers_pool[j] for j in idx_signal]
                else:
                    current_signal = []

                genes_std = current_noise + current_signal
                rng.shuffle(genes_std)

                # Curated gene list
                genes_cur_filtered = [g for g in genes_std if g not in mask_reasons_global]
                genes_cur = genes_cur_filtered[:]

                if len(genes_cur) < n_markers:
                    needed = n_markers - len(genes_cur)
                    refill_candidates = [g for g in true_markers_pool if g not in genes_cur]

                    if needed > len(refill_candidates) and len(true_markers_pool) > 0:
                        extra_needed = needed - len(refill_candidates)
                        extra_idx = rng.choice(
                            len(true_markers_pool),
                            size=extra_needed,
                            replace=True,
                        )
                        refill_candidates.extend([true_markers_pool[j] for j in extra_idx])

                    genes_cur.extend(refill_candidates[:needed])

                genes_cur = genes_cur[:n_markers]

                payload_std = {
                    "panel": "C",
                    "method": "Standard",
                    "dataset": "CD8",
                    "cluster_id": str(cluster_name),
                    "ground_truth": str(gt_label),
                    "noise_ratio": float(ratio),
                    "repeat_id": int(rep),
                    "n_genes": int(n_markers),
                    "use_auto_context": bool(STANDARD_USE_AUTO_CONTEXT_FOR_FIG4),
                    "genes": list(genes_std),
                    "model_name": MODEL_NAME,
                }

                payload_cur = {
                    "panel": "C",
                    "method": "LLM-scCurator",
                    "dataset": "CD8",
                    "cluster_id": str(cluster_name),
                    "ground_truth": str(gt_label),
                    "noise_ratio": float(ratio),
                    "repeat_id": int(rep),
                    "n_genes": int(n_markers),
                    "use_auto_context": bool(CURATOR_USE_AUTO_CONTEXT_FOR_FIG4),
                    "genes": list(genes_cur),
                    "model_name": MODEL_NAME,
                }

                std_key = cache_key(payload_std)
                cur_key = cache_key(payload_cur)

                std_cache_path = cache_dir / f"{std_key}.json"
                cur_cache_path = cache_dir / f"{cur_key}.json"

                std_hit = std_cache_path.exists()
                cur_hit = cur_cache_path.exists()

                rows.append(
                    {
                        "Cluster_ID": str(cluster_name),
                        "Ground_Truth": str(gt_label),
                        "Repeat_ID": int(rep),
                        "Noise_Ratio": float(ratio),
                        "Noise_Pct": float(ratio) * 100.0,
                        "N_Genes": int(n_markers),

                        "Std_Cache_Hit": bool(std_hit),
                        "Cur_Cache_Hit": bool(cur_hit),

                        "Std_Cache_File": std_cache_path.name,
                        "Cur_Cache_File": cur_cache_path.name,

                        "N_Noise_Std": int(sum(g in noise_pool_set for g in genes_std)),
                        "N_Noise_Cur": int(sum(g in noise_pool_set for g in genes_cur)),
                    }
                )

                print(
                    f"    rep {rep+1}/{n_repeats} | "
                    f"Std={'HIT' if std_hit else 'MISS'} | "
                    f"Cur={'HIT' if cur_hit else 'MISS'}"
                )

    df_plan = pd.DataFrame(rows)

    summary = {
        "total_rows": len(df_plan),
        "std_hits": int(df_plan["Std_Cache_Hit"].sum()),
        "std_misses": int((~df_plan["Std_Cache_Hit"]).sum()),
        "cur_hits": int(df_plan["Cur_Cache_Hit"].sum()),
        "cur_misses": int((~df_plan["Cur_Cache_Hit"]).sum()),
        "all_hit_rows": int((df_plan["Std_Cache_Hit"] & df_plan["Cur_Cache_Hit"]).sum()),
        "any_miss_rows": int((~(df_plan["Std_Cache_Hit"] & df_plan["Cur_Cache_Hit"])).sum()),
    }

    print("\n================ CACHE PLAN SUMMARY ================")
    for k, v in summary.items():
        print(f"{k}: {v}")

    miss_df = df_plan.loc[
        ~(df_plan["Std_Cache_Hit"] & df_plan["Cur_Cache_Hit"])
    ].copy()

    if not miss_df.empty:
        print("\nRows with any cache miss (first 20):")
        print(
            miss_df[
                [
                    "Cluster_ID", "Ground_Truth", "Repeat_ID",
                    "Noise_Ratio", "Std_Cache_Hit", "Cur_Cache_Hit"
                ]
            ].head(20).to_string(index=False)
        )
    else:
        print("\nAll rows are full cache hits.")

    return df_plan, summary

df_plan_c, summary_plan_c = inspect_panel_c_cache_plan(
    adata=adata,
    cluster_meta=cluster_meta_c,
    cache_dir=CACHE_DIR,
    noise_ratios=NOISE_RATIOS,
    n_markers=N_MARKERS_NOISE,
    n_repeats=N_REPEATS_NOISE,
    random_seed=RANDOM_SEED,
)

[INFO] Noise pool size: 1235
[INFO] Noise pool examples: ['IGKV2D-24', 'HLA-G', 'HIST1H2BN', 'TRBJ1-2', 'LINC00240', 'HSPB11', 'LINC00601', 'HIST2H2AA4', 'H2AFZ', 'LINC01354']
[CHECK] Global masked genes: 1255

[CHECK] Cluster [1/9]: CD8.c01.Tn.MAL (GT=CD8_Naive)
  --- Noise ratio: 0% ---
    rep 1/5 | Std=HIT | Cur=HIT
    rep 2/5 | Std=HIT | Cur=HIT
    rep 3/5 | Std=HIT | Cur=HIT
    rep 4/5 | Std=HIT | Cur=HIT
    rep 5/5 | Std=HIT | Cur=HIT
  --- Noise ratio: 20% ---
    rep 1/5 | Std=HIT | Cur=HIT
    rep 2/5 | Std=HIT | Cur=HIT
    rep 3/5 | Std=HIT | Cur=HIT
    rep 4/5 | Std=HIT | Cur=HIT
    rep 5/5 | Std=HIT | Cur=HIT
  --- Noise ratio: 40% ---
    rep 1/5 | Std=HIT | Cur=HIT
    rep 2/5 | Std=HIT | Cur=HIT
    rep 3/5 | Std=HIT | Cur=HIT
    rep 4/5 | Std=HIT | Cur=HIT
    rep 5/5 | Std=HIT | Cur=HIT
  --- Noise ratio: 60% ---
    rep 1/5 | Std=HIT | Cur=HIT
    rep 2/5 | Std=HIT | Cur=HIT
    rep 3/5 | Std=HIT | Cur=HIT
    rep 4/5 | Std=HIT | Cur=HIT
    rep 5/5 | Std=HIT

In [76]:
RUN_PANEL_A_LLM = False
RUN_PANEL_C_LLM = True

RUN_REEVALUATION = False
RUN_PLOTTING = False

FORCE_RERUN_PANEL_C = True
FORCE_RERUN_PANEL_A = False

if RUN_PANEL_C_LLM:
    df_panel_c_raw = generate_panel_c_raw_outputs(
        adata=adata,
        curator=curator,
        cluster_meta=cluster_meta_c,
        out_csv=PANEL_C_RAW_CSV,
        noise_ratios=NOISE_RATIOS,
        n_markers=N_MARKERS_NOISE,
        n_repeats=N_REPEATS_NOISE,
        random_seed=RANDOM_SEED,
        force_rerun=FORCE_RERUN_PANEL_C,
    )

[INFO] Noise pool size: 1235
[INFO] Noise pool examples: ['IGKV2D-24', 'HLA-G', 'HIST1H2BN', 'TRBJ1-2', 'LINC00240', 'HSPB11', 'LINC00601', 'HIST2H2AA4', 'H2AFZ', 'LINC01354']

[Panel C] Global masked genes: 1255

[Panel C] Cluster [1/9]: CD8.c01.Tn.MAL (GT=CD8_Naive)
  --- Noise ratio: 0% ---
    rep 1/5 | noise std=0 cur=0
    rep 2/5 | noise std=0 cur=0
    rep 3/5 | noise std=0 cur=0
    rep 4/5 | noise std=0 cur=0
    rep 5/5 | noise std=0 cur=0
  --- Noise ratio: 20% ---
    rep 1/5 | noise std=10 cur=0
    rep 2/5 | noise std=10 cur=0
    rep 3/5 | noise std=10 cur=0
    rep 4/5 | noise std=10 cur=0
    rep 5/5 | noise std=10 cur=0
  --- Noise ratio: 40% ---
    rep 1/5 | noise std=20 cur=0
    rep 2/5 | noise std=20 cur=0
    rep 3/5 | noise std=20 cur=0
    rep 4/5 | noise std=20 cur=0
    rep 5/5 | noise std=20 cur=0
  --- Noise ratio: 60% ---
    rep 1/5 | noise std=30 cur=0
    rep 2/5 | noise std=30 cur=0
    rep 3/5 | noise std=30 cur=0
    rep 4/5 | noise std=30 cur=0
  

In [77]:
deleted, kept, failed = delete_error_cache_only(CACHE_DIR)

[OK] Deleted error cache files: 0
[OK] Kept non-error cache files: 620


In [91]:
# =============================================================================
# 11. Reevaluation step
# =============================================================================

# RUN_REEVALUATION = True
# RUN_PLOTTING = True
df_panel_ab = pd.DataFrame()
df_panel_cd = pd.DataFrame()

if RUN_REEVALUATION:
    if PANEL_A_RAW_CSV.exists():
        df_panel_ab = reevaluate_panel_a_from_saved_answers(
            input_csv=PANEL_A_RAW_CSV,
            output_csv=PANEL_A_REEVAL_CSV,
        )
    elif PANEL_A_REEVAL_CSV.exists():
        df_panel_ab = pd.read_csv(PANEL_A_REEVAL_CSV)
        print(f"[INFO] Loaded existing reevaluated panel A CSV -> {PANEL_A_REEVAL_CSV}")
    else:
        print(f"[WARN] Neither panel A raw nor reevaluated CSV found.")

    if PANEL_C_RAW_CSV.exists():
        df_panel_cd = reevaluate_panel_c_from_saved_answers(
            input_csv=PANEL_C_RAW_CSV,
            output_csv=PANEL_C_REEVAL_CSV,
        )
    elif PANEL_C_REEVAL_CSV.exists():
        df_panel_cd = pd.read_csv(PANEL_C_REEVAL_CSV)
        print(f"[INFO] Loaded existing reevaluated panel C CSV -> {PANEL_C_REEVAL_CSV}")
    else:
        print(f"[WARN] Neither panel C raw nor reevaluated CSV found.")
else:
    if PANEL_A_REEVAL_CSV.exists():
        df_panel_ab = pd.read_csv(PANEL_A_REEVAL_CSV)
        print(f"[INFO] Loaded existing reevaluated panel A CSV -> {PANEL_A_REEVAL_CSV}")
    if PANEL_C_REEVAL_CSV.exists():
        df_panel_cd = pd.read_csv(PANEL_C_REEVAL_CSV)
        print(f"[INFO] Loaded existing reevaluated panel C CSV -> {PANEL_C_REEVAL_CSV}")


[OK] Re-evaluated panel A saved -> /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelA_input_size_robustness_REEVALUATED.csv
[OK] Re-evaluated panel C saved -> /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/panelC_multicluster_noise_simulation_REEVALUATED.csv


In [92]:
# =============================================================================
# 12. Optional sanity check
# =============================================================================
def print_suspicious_rows(df_panel_ab: pd.DataFrame):
    if df_panel_ab.empty:
        print("[WARN] No panel A reevaluated dataframe loaded.")
        return

    wanted = [
        ("CD8.c11.Tex.PDCD1", 50),
        ("CD8.c12.Tex.CXCL13", 50),
        ("CD8.c13.Tex.myl12a", 50),
        ("CD8.c14.Tex.TCF7", 50),
        ("CD8.c15.ISG.IFIT1", 200),
        ("CD8.c17.Tm.NME1", 200),
    ]

    cols = [
        "Cluster_ID", "Ground_Truth", "N_Genes",
        "Std_Answer", "Cur_Answer",
        "Std_PredText_ForScoring", "Cur_PredText_ForScoring",
        "Std_Score_OLD", "Cur_Score_OLD",
        "Std_Score", "Cur_Score", "Score_Diff",
    ]
    cols = [c for c in cols if c in df_panel_ab.columns]

    out = []
    for cid, n in wanted:
        sub = df_panel_ab[
            (df_panel_ab["Cluster_ID"].astype(str) == cid) &
            (df_panel_ab["N_Genes"].astype(int) == int(n))
        ]
        if not sub.empty:
            out.append(sub[cols])

    if len(out) == 0:
        print("[WARN] No suspicious rows matched.")
        return

    debug_df = pd.concat(out, axis=0).reset_index(drop=True)
    print("\n================ SANITY CHECK ROWS ================\n")
    print(debug_df.to_string(index=False))


print_suspicious_rows(df_panel_ab)



================ SANITY CHECK ROWS ================

        Cluster_ID       Ground_Truth  N_Genes                       Std_Answer                       Cur_Answer          Std_PredText_ForScoring          Cur_PredText_ForScoring  Std_Score  Cur_Score  Score_Diff
 CD8.c11.Tex.PDCD1      CD8_Exhausted       50            CD8+ Exhausted T cell            CD8+ Exhausted T cell            CD8+ Exhausted T cell            CD8+ Exhausted T cell        1.0        1.0         0.0
CD8.c12.Tex.CXCL13      CD8_Exhausted       50            CD8+ exhausted T cell            CD8+ Exhausted T cell            CD8+ exhausted T cell            CD8+ Exhausted T cell        1.0        1.0         0.0
CD8.c13.Tex.myl12a      CD8_Exhausted       50            CD8+ exhausted T cell            CD8+ Exhausted T cell            CD8+ exhausted T cell            CD8+ Exhausted T cell        1.0        1.0         0.0
  CD8.c14.Tex.TCF7      CD8_Exhausted       50            Exhausted CD8+ T cell       Effector

In [97]:
# =============================================================================
# 13. Summaries for panels A / B / C / D
# =============================================================================
trace_a = pd.DataFrame()
summary_a = pd.DataFrame()
trace_b = pd.DataFrame()
summary_b = pd.DataFrame()
trace_c = pd.DataFrame()
summary_c = pd.DataFrame()
df_d = pd.DataFrame()
summary_d = pd.DataFrame()

if RUN_PLOTTING and not df_panel_ab.empty:
    # -------------------------
    # Panel A
    # -------------------------
    df_panel_ab["Std_Score_pct"] = maybe_to_percent(df_panel_ab["Std_Score"])
    df_panel_ab["Cur_Score_pct"] = maybe_to_percent(df_panel_ab["Cur_Score"])

    df_ab_long = pd.melt(
        df_panel_ab,
        id_vars=["Cluster_ID", "Ground_Truth", "N_Genes"],
        value_vars=["Std_Score_pct", "Cur_Score_pct"],
        var_name="Method",
        value_name="Sanno_pct",
    )
    df_ab_long["Method"] = df_ab_long["Method"].replace({
        "Std_Score_pct": "Standard",
        "Cur_Score_pct": "LLM-scCurator",
    })

    trace_a = (
        df_ab_long
        .groupby(["Cluster_ID", "Ground_Truth", "Method", "N_Genes"], as_index=False)["Sanno_pct"]
        .mean()
    )
    trace_a.to_csv(OUTPUT_DIR / "Figure4a_cluster_traces.csv", index=False)

    summary_a = summarize_with_ci(
        trace_a,
        group_cols=["Method", "N_Genes"],
        value_col="Sanno_pct",
    )
    summary_a["lower"] = summary_a["lower"].clip(0, 100)
    summary_a["upper"] = summary_a["upper"].clip(0, 100)
    summary_a.to_csv(OUTPUT_DIR / "Figure4a_summary.csv", index=False)

    # -------------------------
    # Panel B
    # -------------------------
    df_b_long = df_ab_long.copy()
    df_b_long["LowConsistency_pct"] = (
        df_b_long["Sanno_pct"] < (FAILURE_THRESHOLD * 100.0)
    ).astype(float) * 100.0

    trace_b = (
        df_b_long
        .groupby(["Cluster_ID", "Ground_Truth", "Method", "N_Genes"], as_index=False)["LowConsistency_pct"]
        .mean()
    )
    trace_b.to_csv(OUTPUT_DIR / "Figure4b_cluster_low_consistency.csv", index=False)

    summary_b = summarize_with_ci(
        trace_b,
        group_cols=["Method", "N_Genes"],
        value_col="LowConsistency_pct",
    )
    summary_b["lower"] = summary_b["lower"].clip(0, 100)
    summary_b["upper"] = summary_b["upper"].clip(0, 100)
    summary_b.to_csv(OUTPUT_DIR / "Figure4b_summary.csv", index=False)

    # -------------------------
    # Panel D
    # -------------------------
    df_d = df_panel_ab.copy()
    df_d["Std_Score_pct"] = maybe_to_percent(df_d["Std_Score"])
    df_d["Cur_Score_pct"] = maybe_to_percent(df_d["Cur_Score"])

    df_d = df_d[
        (df_d["N_Genes"].astype(int) == int(PANEL_D_N_GENES)) &
        (df_d["Ground_Truth"].isin(PANEL_D_HARD_GT_LABELS))
    ].copy()

    if df_d.empty:
        print("[WARN] Panel D data is empty after filtering.")

    df_d["GT_Display"] = df_d["Ground_Truth"].map(
        lambda x: GT_LABEL_DISPLAY.get(x, str(x).replace("CD8_", ""))
    )
    df_d["Delta_pct"] = df_d["Cur_Score_pct"] - df_d["Std_Score_pct"]

    df_d.to_csv(OUTPUT_DIR / "Figure4d_hard_state_defaultN_raw.csv", index=False)

    summary_d = (
        df_d.groupby(["Ground_Truth", "GT_Display"], as_index=False)
        .agg(
            n_clusters=("Cluster_ID", "nunique"),
            Std_Mean_pct=("Std_Score_pct", "mean"),
            Cur_Mean_pct=("Cur_Score_pct", "mean"),
            Delta_Mean_pct=("Delta_pct", "mean"),
        )
    )
    summary_d.to_csv(OUTPUT_DIR / "Figure4d_hard_state_defaultN_summary.csv", index=False)

if RUN_PLOTTING and not df_panel_cd.empty:
    # -------------------------
    # Panel C
    # -------------------------
    df_panel_cd["Standard_Acc_pct"] = maybe_to_percent(df_panel_cd["Standard_Acc"])
    df_panel_cd["Curator_Acc_pct"] = maybe_to_percent(df_panel_cd["Curator_Acc"])
    df_panel_cd["Noise_Pct"] = pd.to_numeric(df_panel_cd["Noise_Ratio"], errors="coerce") * 100.0

    df_cd_acc = pd.melt(
        df_panel_cd,
        id_vars=["Cluster_ID", "Ground_Truth", "Repeat_ID", "Noise_Ratio", "Noise_Pct"],
        value_vars=["Standard_Acc_pct", "Curator_Acc_pct"],
        var_name="Method",
        value_name="Sanno_pct",
    )
    df_cd_acc["Method"] = df_cd_acc["Method"].replace({
        "Standard_Acc_pct": "Standard",
        "Curator_Acc_pct": "LLM-scCurator",
    })

    trace_c = (
        df_cd_acc
        .groupby(["Cluster_ID", "Ground_Truth", "Method", "Noise_Pct"], as_index=False)["Sanno_pct"]
        .mean()
    )
    trace_c.to_csv(OUTPUT_DIR / "Figure4c_cluster_traces.csv", index=False)

    summary_c = summarize_with_ci(
        trace_c,
        group_cols=["Method", "Noise_Pct"],
        value_col="Sanno_pct",
    )
    summary_c["lower"] = summary_c["lower"].clip(0, 100)
    summary_c["upper"] = summary_c["upper"].clip(0, 100)
    summary_c.to_csv(OUTPUT_DIR / "Fig4c_data.csv", index=False)


In [101]:
# =============================================================================
# 14. Plot Fig. 4
# =============================================================================
if RUN_PLOTTING and not df_panel_ab.empty:
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
    plt.rcParams["font.size"] = 11
    plt.rcParams["axes.linewidth"] = 1.3

    COLORS = {
        "Standard": "#5DA5DA",
        "LLM-scCurator": "#D62728",
    }
    MARKERS = {
        "Standard": "s",
        "LLM-scCurator": "o",
    }
    LINESTYLES = {
        "Standard": "--",
        "LLM-scCurator": "-",
    }

    DEFAULT_N = 50 if 50 in sorted(df_panel_ab["N_Genes"].astype(int).unique()) else sorted(df_panel_ab["N_Genes"].astype(int).unique())[0]

    fig = plt.figure(figsize=(13.2, 9.0), dpi=300)
    gs = fig.add_gridspec(2, 2, wspace=0.32, hspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])  # a
    ax2 = fig.add_subplot(gs[0, 1])  # b
    ax3 = fig.add_subplot(gs[1, 0])  # c
    ax4 = fig.add_subplot(gs[1, 1])  # d

    # --------------------------------
    # Panel a: Mean Sanno vs input size
    # --------------------------------
    for method in ["Standard", "LLM-scCurator"]:
        sub_trace = trace_a[trace_a["Method"] == method]
        for cluster_id in sorted(sub_trace["Cluster_ID"].unique()):
            sub_c = sub_trace[sub_trace["Cluster_ID"] == cluster_id].sort_values("N_Genes")
            ax1.plot(
                sub_c["N_Genes"].values,
                sub_c["Sanno_pct"].values,
                color=COLORS[method],
                alpha=0.10,
                linewidth=1.0,
                linestyle=LINESTYLES[method],
                zorder=1,
            )

    for method in ["Standard", "LLM-scCurator"]:
        sub = summary_a[summary_a["Method"] == method].sort_values("N_Genes")
        ax1.fill_between(
            sub["N_Genes"].values,
            sub["lower"].values,
            sub["upper"].values,
            color=COLORS[method],
            alpha=0.18,
            linewidth=0,
            zorder=2,
        )
        ax1.plot(
            sub["N_Genes"].values,
            sub["mean"].values,
            color=COLORS[method],
            linestyle=LINESTYLES[method],
            marker=MARKERS[method],
            linewidth=2.6,
            markersize=6.5,
            label=method,
            zorder=3,
        )

    ax1.axvline(DEFAULT_N, color="gray", linestyle=":", linewidth=1.0, alpha=0.8, zorder=0)
    #ax1.text(
    #    DEFAULT_N, 3,
    #    f"default\nN={DEFAULT_N}",
    #    ha="center", va="bottom",
    #    fontsize=8, color="dimgray"
    #)

    ax1.set_xlabel("Top N genes (input size)", fontweight="bold")
    ax1.set_ylabel("Mean Sanno (%)", fontweight="bold")
    ax1.set_xticks(sorted(df_panel_ab["N_Genes"].astype(int).unique()))
    ax1.set_ylim(-2, 105)
    ax1.set_yticks([0, 20, 40, 60, 80, 100])
    ax1.grid(True, axis="y", linestyle=":", alpha=0.55)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.text(
        0.75, 0.05,
        f"n clusters = {trace_a['Cluster_ID'].nunique()}",
        transform=ax1.transAxes,
        fontsize=9,
        color="dimgray"
    )

    # --------------------------------
    # Panel b: Low-consistency rate
    # --------------------------------
    for method in ["Standard", "LLM-scCurator"]:
        sub_trace = trace_b[trace_b["Method"] == method]
        for cluster_id in sorted(sub_trace["Cluster_ID"].unique()):
            sub_c = sub_trace[sub_trace["Cluster_ID"] == cluster_id].sort_values("N_Genes")
            ax2.plot(
                sub_c["N_Genes"].values,
                sub_c["LowConsistency_pct"].values,
                color=COLORS[method],
                alpha=0.10,
                linewidth=1.0,
                linestyle=LINESTYLES[method],
                zorder=1,
            )

    for method in ["Standard", "LLM-scCurator"]:
        sub = summary_b[summary_b["Method"] == method].sort_values("N_Genes")
        ax2.fill_between(
            sub["N_Genes"].values,
            sub["lower"].values,
            sub["upper"].values,
            color=COLORS[method],
            alpha=0.18,
            linewidth=0,
            zorder=2,
        )
        ax2.plot(
            sub["N_Genes"].values,
            sub["mean"].values,
            color=COLORS[method],
            linestyle=LINESTYLES[method],
            marker=MARKERS[method],
            linewidth=2.6,
            markersize=6.5,
            label=method,
            zorder=3,
        )

    ax2.axvline(DEFAULT_N, color="gray", linestyle=":", linewidth=1.0, alpha=0.8, zorder=0)
    #ax2.text(
    #    DEFAULT_N, 3,
    #    f"default\nN={DEFAULT_N}",
    #    ha="center", va="bottom",
    #    fontsize=8, color="dimgray"
    #)

    ax2.set_xlabel("Top N genes (input size)", fontweight="bold")
    ax2.set_ylabel(f"Low-consistency predictions (%)\n(Sanno < {FAILURE_THRESHOLD})", fontweight="bold")
    ax2.set_xticks(sorted(df_panel_ab["N_Genes"].astype(int).unique()))
    ax2.set_ylim(-2, 105)
    ax2.set_yticks([0, 20, 40, 60, 80, 100])
    ax2.grid(True, axis="y", linestyle=":", alpha=0.55)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    # --------------------------------
    # Panel c: Noise injection
    # --------------------------------
    if not summary_c.empty and not trace_c.empty:
        for method in ["Standard", "LLM-scCurator"]:
            sub_trace = trace_c[trace_c["Method"] == method]
            for cluster_id in sorted(sub_trace["Cluster_ID"].unique()):
                sub_c = sub_trace[sub_trace["Cluster_ID"] == cluster_id].sort_values("Noise_Pct")
                ax3.plot(
                    sub_c["Noise_Pct"].values,
                    sub_c["Sanno_pct"].values,
                    color=COLORS[method],
                    alpha=0.10,
                    linewidth=1.0,
                    linestyle=LINESTYLES[method],
                    zorder=1,
                )

        for method in ["Standard", "LLM-scCurator"]:
            sub = summary_c[summary_c["Method"] == method].sort_values("Noise_Pct")
            ax3.fill_between(
                sub["Noise_Pct"].values,
                sub["lower"].values,
                sub["upper"].values,
                color=COLORS[method],
                alpha=0.18,
                linewidth=0,
                zorder=2,
            )
            ax3.plot(
                sub["Noise_Pct"].values,
                sub["mean"].values,
                color=COLORS[method],
                linestyle=LINESTYLES[method],
                marker=MARKERS[method],
                linewidth=2.6,
                markersize=6.5,
                label=method,
                zorder=3,
            )

        ax3.set_xlabel("Fraction of biological-noise genes (%)", fontweight="bold")
        ax3.set_ylabel("Mean Sanno (%)", fontweight="bold")
        ax3.set_xlim(-5, 85)
        ax3.set_xticks([0, 20, 40, 60, 80])
        ax3.set_ylim(-1, 105)
        ax3.set_yticks([0, 20, 40, 60, 80, 100])
        ax3.grid(True, axis="y", linestyle=":", alpha=0.55)
        ax3.spines["top"].set_visible(False)
        ax3.spines["right"].set_visible(False)
        ax3.text(
            0.02, 0.05,
            f"n clusters = {trace_c['Cluster_ID'].nunique()},\nrepeats = {N_REPEATS_NOISE}",
            transform=ax3.transAxes,
            fontsize=9,
            color="dimgray"
        )
    else:
        ax3.axis("off")
        ax3.text(0.5, 0.5, "Panel c unavailable", ha="center", va="center")

    # --------------------------------
    # Panel d: Ambiguity-prone states
    # --------------------------------
    hard_gt_available = [gt for gt in PANEL_D_HARD_GT_LABELS if gt in df_d["Ground_Truth"].unique()]
    x_centers = np.arange(len(hard_gt_available))
    offset_std = -0.14
    offset_cur = 0.14

    for i, gt in enumerate(hard_gt_available):
        sub = df_d[df_d["Ground_Truth"] == gt].sort_values("Cluster_ID").copy()

        for _, row in sub.iterrows():
            xs = [i + offset_std, i + offset_cur]
            ys = [row["Std_Score_pct"], row["Cur_Score_pct"]]
            ax4.plot(
                xs, ys,
                color="0.75",
                linewidth=1.1,
                alpha=0.9,
                zorder=1,
            )

            ax4.scatter(
                i + offset_std,
                row["Std_Score_pct"],
                color=COLORS["Standard"],
                edgecolor="black",
                linewidth=0.4,
                s=38,
                marker=MARKERS["Standard"],
                zorder=2,
            )
            
            ax4.scatter(
                i + offset_cur,
                row["Cur_Score_pct"],
                color=COLORS["LLM-scCurator"],
                edgecolor="black",
                linewidth=0.4,
                s=38,
                marker=MARKERS["LLM-scCurator"],
                zorder=2,
            )

        ax4.scatter(
            i + offset_std,
            sub["Std_Score_pct"].mean(),
            color=COLORS["Standard"],
            edgecolor="black",
            linewidth=0.9,
            s=95,
            marker=MARKERS["Standard"],
            zorder=4,
        )
        ax4.scatter(
            i + offset_cur,
            sub["Cur_Score_pct"].mean(),
            color=COLORS["LLM-scCurator"],
            edgecolor="black",
            linewidth=0.9,
            s=95,
            marker=MARKERS["LLM-scCurator"],
            zorder=4,
        )

        mean_delta = sub["Delta_pct"].mean()
        ax4.text(
            i,
            min(102, max(sub["Std_Score_pct"].max(), sub["Cur_Score_pct"].max()) + 5),
            f"{mean_delta:+.1f}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="dimgray",
        )

    ax4.set_xticks(x_centers)
    ax4.set_xticklabels([GT_LABEL_DISPLAY.get(gt, gt.replace("CD8_", "")) for gt in hard_gt_available])
    ax4.set_xlabel(f"Predefined ambiguity-prone states (N={PANEL_D_N_GENES})", fontweight="bold")
    ax4.set_ylabel("Cluster-level Sanno (%)", fontweight="bold")
    ax4.set_ylim(-2, 105)
    ax4.set_yticks([0, 20, 40, 60, 80, 100])
    ax4.grid(True, axis="y", linestyle=":", alpha=0.55)
    ax4.spines["top"].set_visible(False)
    ax4.spines["right"].set_visible(False)
    ax4.text(
        0.65, 0.05,
        f"hard states pre-specified;\nn clusters = {df_d['Cluster_ID'].nunique()}",
        transform=ax4.transAxes,
        fontsize=9,
        color="dimgray"
    )

    handles = [
        plt.Line2D([0], [0], color=COLORS["Standard"], linestyle=LINESTYLES["Standard"], marker=MARKERS["Standard"], linewidth=2.6, markersize=6.5, label="Standard\n(top DEGs))"),
        plt.Line2D([0], [0], color=COLORS["LLM-scCurator"], linestyle=LINESTYLES["LLM-scCurator"], marker=MARKERS["LLM-scCurator"], linewidth=2.6, markersize=6.5, label="Full pipeline\n(LLM-scCurator)"),
    ]
    fig.legend(
        handles,
        ["Standard\n(top DEGs))", "Full pipeline\n(LLM-scCurator)"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.97),
        ncol=2,
        frameon=False,
        fontsize=14,
        handlelength=2.5,
    )

    ax1.text(-0.14, 1.04, "a", transform=ax1.transAxes, fontsize=14, fontweight="bold")
    ax2.text(-0.14, 1.04, "b", transform=ax2.transAxes, fontsize=14, fontweight="bold")
    ax3.text(-0.14, 1.04, "c", transform=ax3.transAxes, fontsize=14, fontweight="bold")
    ax4.text(-0.14, 1.04, "d", transform=ax4.transAxes, fontsize=14, fontweight="bold")

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    png_path = OUTPUT_DIR / "Fig4a_d.png"
    pdf_path = OUTPUT_DIR / "Fig4a_d.pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(f"\n[OK] Figure saved: {png_path}")
    print(f"[OK] Figure saved: {pdf_path}")


[OK] Figure saved: /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/Figure4_Robustness_CD8_4panel.png
[OK] Figure saved: /work/paper/gb_resubmission/output/fig4_robustness_cd8_4panel/Figure4_Robustness_CD8_4panel.pdf
